### 0. Import required libraries

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

### 1. Import un-sampled dataset

In [ ]:
dataset_path = 'US_Accidents_March23_sampled_500k.csv'
df = pd.read_csv(dataset_path)

### 2. Exploratory Data Analysis

In [ ]:
# Get a quick overview of the dataset
print(df.info())

In [ ]:
severity_counts = df['Severity'].value_counts().sort_index()
severity_pct = (severity_counts / len(df) * 100).round(2)
ax = severity_counts.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_xlabel('Accident Severity')
ax.set_ylabel('Count')
ax.set_title('Distribution of Accident Severity')
for i, (cnt, pct) in enumerate(zip(severity_counts, severity_pct)):
    ax.text(i, cnt, f'{pct}%', ha='center', va='bottom')
plt.tight_layout()
plt.show()

As mentioned in our proposal, severity is heavily imbalanced, with severity 2 making up roughly 80% of the records. 

In [ ]:
missing = df.isna().sum()
missing = pd.DataFrame({
    'missing': missing,
    'percent': (missing / len(df) * 100).round(2)
})
missing = missing[missing['missing'] > 0].sort_values('percent', ascending=False)
missing

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(missing.index[::-1], missing['percent'][::-1])
ax.set_xlabel('Missing values (%)')
ax.set_title('Missingness by feature')
plt.tight_layout()
plt.show()

22 of the columns have missing values. `End_Lat` and `End_Lng` are the worst, missing in about 44% of records. `Precipitation(in)` (28.5%) and `Wind_Chill(F)` (25.8%) are the next most incomplete. The other weather features are missing in only about 2 to 7% of records. 

`Percipitation` is missing about 30% of the time, however, as you can see from the data below, missing values occured almost entirely under dry or cloudy conditions, so we believe it is safe to treat them as 0.

In [ ]:
missing_precip = df['Precipitation(in)'].isna()
print("Weather conditions where precipitation is missing:")
print(df.loc[missing_precip, 'Weather_Condition'].value_counts(normalize=True).head(10).mul(100).round(1))
print("\nWeather conditions where precipitation is recorded:")
print(df.loc[~missing_precip, 'Weather_Condition'].value_counts(normalize=True).head(10).mul(100).round(1))

In [ ]:
num_cols = ['Distance(mi)', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
            'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, col in zip(axes.ravel(), num_cols):
    data = df[col].dropna()
    ax.hist(data, bins=50, edgecolor='white')
    ax.set_title(f'{col}\nmax={data.max():.1f}, p99.5={data.quantile(0.995):.1f}')
    ax.set_ylabel('Count')
for ax in axes.ravel()[len(num_cols):]:
    ax.axis('off')
plt.suptitle('Distributions of numeric features (unclipped)', y=1.02)
plt.tight_layout()
plt.show()

As you can see from the graphs above, much of the numerical data have outlier values, which cause the graphs to appear zoomed-out, and will later lead to issues when running our data mining algorithms. To help with this, during our data preprocessing we will remove the extreme outliers and keep only the data within the 99.5 percentile.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['State'].value_counts().head(15).plot(kind='barh', ax=axes[0])
axes[0].invert_yaxis()
axes[0].set_title('Top 15 states by accident count')

df['Weather_Condition'].value_counts().head(15).plot(kind='barh', ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title('Top 15 weather conditions')

plt.tight_layout()
plt.show()

Understandably, the more populated states have more counts within the dataset. It is important to keep in mind that we are not predicting accident *probability* but rather accident *severity*, so the imbalance should not neccessarily bias the Severity target. However, the Severity of an accident still may vary by state regardless. 

In [ ]:
# Which boolean flags are rarely True?
bool_cols = df.select_dtypes(include='bool').columns
percent_true = (df[bool_cols].mean() * 100).round(2).sort_values()
print("Percent of records where each flag is True:")
print(percent_true)

# Flags that are True in under 1% of records
rare_flags = percent_true[percent_true < 1].index

# Do those rare flags show up more often in severe accidents (severity 3 or 4)?
high = df['Severity'] >= 3
print(f"\nOverall percent of accidents with severity 3 or 4: {high.mean() * 100:.1f}%")

# For each rare flag, the percent of its True rows that are severity 3 or 4
print("\nPercent severity 3 or 4 when the flag is True:")
print(df[rare_flags].apply(lambda c: high[c].mean() * 100).round(1))

Binary flags that are true in <1% of records (e.g., Roundabout, Turning_Loop, Bump, etc.) will be dropped, as their rates of severe acidents (severity 3 or 4) were at or below the overall rate of 19.5%, so they carry little information about severity.

In [ ]:
print(f"Total Unique Weather_Condition Values: {df['Weather_Condition'].nunique()}")
print((df['Weather_Condition'].value_counts(normalize=True).head(25) * 100).round(2))
shares = df['Weather_Condition'].value_counts(normalize=True).cumsum() * 100
n_99 = (shares < 99).sum() + 1
print(f"The top {n_99} weather conditions cover 99% of records with a weather value")

As you can see, the `Weather_Condition` feature has 108 unique values, however, the top 27 conditions cover 99% of the records. Therefore, to reduce the number of category options, we can keep the top 27 conditions, and put the rest into an "other" option.

### 3. Data Transformation & Preprocessing

We start our data transformation by parsing the string timestamps into numerical data:

In [ ]:
# Create new features

# Convert the String timestamps into pandas datetime objects
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed', errors='coerce')
df['End_Time'] = pd.to_datetime(df['End_Time'], format='mixed', errors='coerce')

# Create new features using the datetime objects
df['Hour'] = df['Start_Time'].dt.hour
df['DayOfWeek'] = df['Start_Time'].dt.dayofweek
df['Month'] = df['Start_Time'].dt.month
df['Duration_min'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60

As mentioned above, we will also assume that no percipitation is zero percipitation:

In [ ]:
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(0)

In [ ]:
# Removal of features which are not needed
to_drop = [
    'Country', # always US
    'ID', # unique per-row, so useless for training
    'Turning_Loop', # always False
    'Roundabout', # almost always False
    'Bump', # almost always False
    'Traffic_Calming', # almost always False
    'No_Exit', # almost always False, severe rate below overall
    'Give_Way', # almost always False, severe rate below overall
    'Railway', # almost always False, severe rate below overall
    'Airport_Code', # unrelated?
    'Timezone', # implied by State
    'Source', # source of the data not needed for predictions
    'Wind_Chill(F)', # redundant with Temperature
    'Weather_Timestamp', # when the weather was recorded, redundant with Start_Time
    'Wind_Direction', # likely irrelevant
    
    'Civil_Twilight', # nearly identical to Sunrise_Sunset
    'Nautical_Twilight', # nearly identical to Sunrise_Sunset
    'Astronomical_Twilight', # nearly identical to Sunrise_Sunset
    
    'Zipcode', # too many unique values, location already covered by Start_Lat/Start_Lng
    'County', # too many unique values, location already covered by State and coordinates
    'End_Lat', # 44% missing and close to Start_Lat
    'End_Lng', # 44% missing and close to Start_Lng
    'Street', # too many unique values to encode usefully
    'City', # too many unique values to encode usefully
    
    'Description', # TODO: perhaps we *can* use this by making a vector?
    
    'Start_Time', # replaced by Hour, DayOfWeek, and Month
    'End_Time', # only needed to compute Duration_min
]

df = df.drop(columns=[c for c in to_drop if c in df.columns])

This leaves us with the following columns:

In [ ]:
print(df.columns)

As reasoned for above, we will keep the top 27 options for `Weather_Condition`, and put the rest into an "other" option:

In [ ]:
top_conditions = df['Weather_Condition'].value_counts().head(27).index
df['Weather_Condition'] = df['Weather_Condition'].where(
    df['Weather_Condition'].isin(top_conditions) | df['Weather_Condition'].isna(),
    'Other'
)

Finally, as we have an abundance of data, we will simply drop any records with missing data:

In [ ]:
rows_before = len(df)
df = df.dropna()
print(f"Dropped {rows_before - len(df)} rows ({(rows_before - len(df)) / rows_before * 100:.1f}%), {len(df)} remaining")
print(df['Severity'].value_counts(normalize=True).sort_index().round(4) * 100)